# P 侧 DRAM 容量扫描
固定 SSD 128× 和读写带宽，复用 8192 块对照。

In [ ]:
from pathlib import Path
import json, re
import pandas as pd
import matplotlib.pyplot as plt

results = Path.home()/'project/results'
batch = results/'glm52-1p1d-p-g3-dram/20261106T034430.383604Z'
baseline = results/'glm52-1p1d-p-g3-capacity/20261106T012822.616008Z/p_g3_local_128x/20261106T030805.664037Z'
out = Path.home()/'project/aisimulate/experiments/disk/analysis/p_g3_local_dram_sweep/assets'
out.mkdir(exist_ok=True)
rows = []
for blocks in (2048, 4096, 8192, 32768):
    run = baseline if blocks == 8192 else next((batch/f'p_g3_local_dram_{blocks}').glob('*/run.json')).parent
    meta = json.loads((run/'run.json').read_text())
    assert meta['status']=='completed'
    report = run/'prediction/prediction.json'
    with report.open() as f:
        # These summary fields precede the large per-request array.
        header = f.read(25000)
    def metric(name):
        return float(re.search(r'"'+name+r'"\s*:\s*([0-9.eE+-]+)',header)[1])
    g3 = json.JSONDecoder().raw_decode(header[re.search(r'"g3_offload"\s*:\s*',header).end():])[0]
    completed = int(metric('completed_requests'))
    assert completed == meta['selected_requests'] == 665415
    good = int(metric('goodput_completed_requests'))
    rows.append(dict(dram_blocks=blocks,completed=completed,sla_pass=good,sla_percent=100*good/completed,
                     mean_ttft_s=metric('mean_ttft_ms')/1000,p95_ttft_s=metric('p95_ttft_ms')/1000,
                     p99_ttft_s=metric('p99_ttft_ms')/1000,max_gap_s=metric('max_itl_ms')/1000,
                     ssd_read_tb=g3['read']['completed_bytes']/1e12,ssd_write_tb=g3['write']['completed_bytes']/1e12,
                     g3_bypassed_restores=g3.get('bypassed_restores',0),
                     trace_sha256=meta['selected_trace_sha256'],report=str(report)))
assert len({row['trace_sha256'] for row in rows})==1
frame = pd.DataFrame(rows)
frame.to_csv(out/'metrics.csv',index=False)
(out/'summary.json').write_text(json.dumps(rows,indent=2))
print(frame.drop(columns=['report','trace_sha256']).to_string(index=False))
fig, axes = plt.subplots(1,3,figsize=(12,3.8))
for ax,key,label in zip(axes,['sla_percent','ssd_read_tb','max_gap_s'],
                       ['SLA pass (%)','SSD read (TB per TP shard)','Max token gap (s)']):
    ax.plot(frame.dram_blocks.astype(str),frame[key],marker='o')
    ax.set(xlabel='DRAM blocks',ylabel=label)
    ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig(out/'comparison.png',dpi=160)
plt.show()
